In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 1000
mu = 0.0005
sigma = 0.01

prices = [100]

for i in range(n):
    shock = np.random.normal(mu, sigma)
    price = prices[-1] * np.exp(shock)
    prices.append(price)

df = pd.DataFrame()
df["close"] = prices

In [2]:
df["open"] = df["close"].shift(1)
df["high"] = df[["open","close"]].max(axis=1) * (1 + np.random.rand(len(df))*0.01)
df["low"] = df[["open","close"]].min(axis=1) * (1 - np.random.rand(len(df))*0.01)
df["volume"] = np.random.randint(100,1000,len(df))
df = df.dropna()

In [3]:
df["log_return"] = np.log(df["close"]/df["close"].shift(1))

In [4]:
df["volatility"] = df["log_return"].rolling(20).std()

In [5]:
df["tr"] = df["high"] - df["low"]
df["atr"] = df["tr"].rolling(14).mean()

In [6]:
df["momentum"] = df["close"] - df["close"].shift(10)

In [7]:
df["volume_spike"] = df["volume"]/df["volume"].rolling(20).mean()

In [8]:
df["future_vol"] = df["log_return"].rolling(10).std().shift(-10)

threshold = df["future_vol"].quantile(0.7)

df["target"] = (df["future_vol"] > threshold).astype(int)

In [9]:
def monte_carlo_volatility(price, sigma, steps=10, sims=500):

    results = []

    for _ in range(sims):

        path = price

        returns = []

        for i in range(steps):

            shock = np.random.normal(0, sigma)

            path = path * np.exp(shock)

            returns.append(shock)

        vol = np.std(returns)

        results.append(vol)

    return np.mean(results)

In [10]:
df["mc_vol"] = df["close"].apply(lambda x: monte_carlo_volatility(x,0.01))

In [11]:
features = [
"log_return",
"volatility",
"atr",
"momentum",
"volume_spike",
"mc_vol"
]

In [12]:
df = df.dropna()

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = df[features]
y = df["target"]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6
)

model.fit(X_train,y_train)

RandomForestClassifier(max_depth=6, n_estimators=200)

In [14]:
probs = model.predict_proba(X_test)

df_probs = probs[:,1]

In [15]:
from sklearn.metrics import accuracy_score,roc_auc_score

pred = model.predict(X_test)

print("Accuracy:",accuracy_score(y_test,pred))
print("AUC:",roc_auc_score(y_test,df_probs))

Accuracy: 0.7319587628865979
AUC: 0.5890580008227067


# **QTM**

In [16]:
pip install pennylane numpy pandas scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 92.1 MB/s eta 0:00:00


In [20]:
X = df[features]

In [18]:
import numpy as np

def normalize_features(x):
    x = np.array(x)
    return (x - x.min()) / (x.max() - x.min())

In [19]:
import pennylane as qml

n_qubits = 4

dev = qml.device("default.qubit", wires=n_qubits)

In [21]:
@qml.qnode(dev)
def quantum_circuit(features):

    # Feature encoding
    for i in range(n_qubits):
        qml.RY(features[i] * np.pi, wires=i)

    # Entanglement
    qml.CNOT(wires=[0,1])
    qml.CNOT(wires=[1,2])
    qml.CNOT(wires=[2,3])

    # Nonlinear rotations
    for i in range(n_qubits):
        qml.RZ(features[i] * np.pi, wires=i)

    # Measure probability of |1>
    return qml.probs(wires=0)

In [22]:
features = [0.6, 0.7, 0.3, 0.5]

features = normalize_features(features)

probs = quantum_circuit(features)

Pq = probs[1]

print("Quantum Volatility Probability:", Pq)

Quantum Volatility Probability: 0.8535533905932737


In [23]:
Pc = 0.59
alpha = 0.7

Ph = alpha * Pc + (1-alpha) * Pq

print("Hybrid Probability:", Ph)

Hybrid Probability: 0.6690660171779821


In [24]:
print(qml.draw(quantum_circuit)(features))

0: ──RY(2.36)─╭●──RZ(2.36)─────────────────────┤  Probs
1: ──RY(3.14)─╰X─╭●─────────RZ(3.14)───────────┤       
2: ──RY(0.00)────╰X────────╭●─────────RZ(0.00)─┤       
3: ──RY(1.57)──────────────╰X─────────RZ(1.57)─┤       


In [25]:
import pennylane as qml
import numpy as np

# Number of qubits (features)
n_qubits = 4

# Quantum simulator
dev = qml.device("default.qubit", wires=n_qubits)

# Normalize features to [0,1]
def normalize_features(x):
    x = np.array(x)
    return (x - np.min(x)) / (np.max(x) - np.min(x) + 1e-8)

@qml.qnode(dev)
def quantum_circuit(features):

    # Feature Encoding
    for i in range(n_qubits):
        qml.RY(features[i] * np.pi, wires=i)

    # Entanglement Layer
    qml.CNOT(wires=[0,1])
    qml.CNOT(wires=[1,2])
    qml.CNOT(wires=[2,3])

    # Nonlinear Phase Rotations
    for i in range(n_qubits):
        qml.RZ(features[i] * np.pi, wires=i)

    # Return probability of measuring |0> or |1> on qubit 0
    return qml.probs(wires=0)


# Example classical features
features = [0.6, 0.7, 0.0, 0.5]

# Normalize
features = normalize_features(features)

# Run circuit
probs = quantum_circuit(features)

# Quantum probability of High Volatility
Pq = probs[1]

print("Quantum Volatility Probability:", Pq)

# Print circuit
print(qml.draw(quantum_circuit)(features))

Quantum Volatility Probability: 0.9504844256057849
0: ──RY(2.69)─╭●──RZ(2.69)─────────────────────┤  Probs
1: ──RY(3.14)─╰X─╭●─────────RZ(3.14)───────────┤       
2: ──RY(0.00)────╰X────────╭●─────────RZ(0.00)─┤       
3: ──RY(2.24)──────────────╰X─────────RZ(2.24)─┤       
